In [ ]:
# Install required libraries for the lab.
%pip install openai voyageai numpy scikit-learn

import os
import numpy as np
from openai import OpenAI
import voyageai
from IPython.display import Markdown, display

# Replace these with your real API keys before running.
OPENAI_API_KEY = ""
VOYAGE_API_KEY = ""

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["VOYAGE_API_KEY"] = VOYAGE_API_KEY


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Phase 1: Domain Vocabulary Trap

This phase is used to test whether embedding models can recognize that a layman-style health query and a clinically worded sentence describe the same underlying medical situation. The goal is to compare a general embedding model against a second model and see which one better bridges specialized vocabulary.

## Step 1: Define Sentences

In [7]:
# Define the document and query sentences.
document_sentence = "The patient exhibited severe diaphoresis and tachycardia"
query_sentence = "The person was sweating heavily with a fast heart rate"

print("Document sentence:", document_sentence)
print("Query sentence:", query_sentence)


Document sentence: The patient exhibited severe diaphoresis and tachycardia
Query sentence: The person was sweating heavily with a fast heart rate


## Step 2: General Model (OpenAI)

In [8]:
# Generate OpenAI embeddings and compute cosine similarity manually.
openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document_sentence, query_sentence],
)

openai_document_embedding = np.array(openai_response.data[0].embedding)
openai_query_embedding = np.array(openai_response.data[1].embedding)
openai_similarity = np.dot(openai_document_embedding, openai_query_embedding) / (
    np.linalg.norm(openai_document_embedding) * np.linalg.norm(openai_query_embedding)
)

print(f"OpenAI text-embedding-3-small similarity: {openai_similarity:.4f}")


OpenAI text-embedding-3-small similarity: 0.6391


## Step 3: Domain-Specific Model (Voyage AI)

In [9]:
# Try Voyage models in order and use the first one that succeeds.
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)
voyage_model_candidates = ["voyage-3", "voyage-finance-2", "voyage-law-2"]
voyage_model_used = None
voyage_response = None
voyage_errors = []

for candidate_model in voyage_model_candidates:
    try:
        voyage_response = voyage_client.embed(
            texts=[document_sentence, query_sentence],
            model=candidate_model,
            input_type="document",
        )
        voyage_model_used = candidate_model
        break
    except Exception as error:
        voyage_errors.append(f"{candidate_model}: {error}")

if voyage_response is None:
    raise RuntimeError("All Voyage model attempts failed:\n" + "\n".join(voyage_errors))

voyage_document_embedding = np.array(voyage_response.embeddings[0])
voyage_query_embedding = np.array(voyage_response.embeddings[1])
voyage_similarity = np.dot(voyage_document_embedding, voyage_query_embedding) / (
    np.linalg.norm(voyage_document_embedding) * np.linalg.norm(voyage_query_embedding)
)

print(f"Voyage model used: {voyage_model_used}")
print(f"Voyage similarity: {voyage_similarity:.4f}")


Voyage model used: voyage-3
Voyage similarity: 0.8185


## Step 4: Comparison Table

In [10]:
# Compare the two models side by side.
if openai_similarity > voyage_similarity:
    openai_verdict = "Better"
    voyage_verdict = "Lower"
elif voyage_similarity > openai_similarity:
    openai_verdict = "Lower"
    voyage_verdict = "Better"
else:
    openai_verdict = "Tie"
    voyage_verdict = "Tie"

comparison_table = f"""
| Model | Similarity Score | Verdict |
|---|---:|---|
| OpenAI text-embedding-3-small | {openai_similarity:.4f} | {openai_verdict} |
| Voyage {voyage_model_used} | {voyage_similarity:.4f} | {voyage_verdict} |
"""

display(Markdown(comparison_table))



| Model | Similarity Score | Verdict |
|---|---:|---|
| OpenAI text-embedding-3-small | 0.6391 | Lower |
| Voyage voyage-3 | 0.8185 | Better |


## Step 5: Reflection Questions

Q1: Which model successfully mapped the layman query to clinical jargon?

A1: The Voyage model had a higher cosine similarity score, so it mapped the layman query to the clinical wording more effectively in this comparison.



Q2: Why do large generalist models fail in highly specialized domains compared to domain-tuned models?

A2: General models are built to handle a little bit of everything, so they often miss the deeper meaning of specialized terms and expert language. Domain-tuned models are trained on field-specific material, which helps them better understand technical vocabulary, common phrasing, and how concepts relate in that domain.



